# toolpath4 — XYZB 4-Axis Demo Notebook

This notebook demonstrates the full pipeline:
1. Load/generate a test mesh
2. Slice it into horizontal layers
3. Assign B-axis angles based on overhang analysis
4. Preview the toolpath in 3D
5. Export G-code

In [ ]:
import sys, os
# Ensure the package is importable
sys.path.insert(0, os.path.abspath('..'))

from toolpath4 import (
    PrinterConfig, default_config,
    Move, StateChange, StepList, State,
    helix, oscillating_b, circle, polyline, densify,
    tip_to_pivot, pivot_to_tip,
    compile_gcode, dry_run,
    preview_matplotlib,
    Slicer,
)
print('toolpath4 imported successfully')

## 1. Configuration

In [ ]:
cfg = default_config()
issues = cfg.validate_config()
print('Config valid' if not issues else issues)
print(f'Build volume: {cfg.bed_x_max}x{cfg.bed_y_max}x{cfg.bed_z_max} mm')
print(f'B range: [{cfg.b_min_deg}, {cfg.b_max_deg}] deg')
print(f'Steps/deg B: {cfg.steps_per_degree_b:.1f}')

## 2. Procedural helix with oscillating B

In [ ]:
# Generate a helix
h = helix(center=(150, 150), radius=20, z_start=0.3, z_end=30,
          turns=8, b=90, points_per_turn=60, feedrate=3000)

# Add oscillating B angle
h_osc = oscillating_b(start_b=35, end_b=90, frequency=3, phase=0, path=h)

steps_helix = StepList(h_osc)
stats = dry_run(steps_helix, cfg)
print(f'Helix: {stats.move_count} moves, {stats.total_path_length_mm:.0f} mm path')
print(f'B range: [{stats.b_min_deg:.1f}, {stats.b_max_deg:.1f}] deg')

In [ ]:
%matplotlib inline
preview_matplotlib(steps_helix, title='Helix with Oscillating B', show=True)

## 3. Kinematics check

In [ ]:
# Verify tip ↔ pivot round-trip
tip = (150.0, 150.0, 0.0)
for b_deg in [0, 30, 45, 90, -45]:
    pivot = tip_to_pivot(tip, b_deg, cfg)
    tip_back = pivot_to_tip(pivot, b_deg, cfg)
    err = sum((a - b)**2 for a, b in zip(tip, tip_back))**0.5
    print(f'B={b_deg:+4d}°  pivot={tuple(round(v,2) for v in pivot)}  roundtrip err={err:.1e}')

## 4. Generate a test STL and slice it

In [ ]:
import trimesh, tempfile, os

# Create a T-shaped overhang model
column = trimesh.creation.box(extents=[20, 20, 30])
column.apply_translation([150, 150, 15])
shelf = trimesh.creation.box(extents=[40, 20, 5])
shelf.apply_translation([150, 150, 32.5])
mesh = trimesh.util.concatenate([column, shelf])

stl_path = os.path.join(tempfile.gettempdir(), 'overhang_test.stl')
mesh.export(stl_path)
print(f'Test STL: {stl_path} ({len(mesh.faces)} faces)')

In [ ]:
slicer = Slicer(config=PrinterConfig(layer_height=0.5))  # coarse for demo speed
steps = slicer.process(stl_path, transition='smooth')
print(f'\nGenerated {len(steps)} steps ({len(steps.moves())} moves)')

In [ ]:
print(slicer.rotation_schedule_summary())

In [ ]:
%matplotlib inline
preview_matplotlib(steps, title='Overhang Slice — B coloured', show=True)

## 5. Export G-code

In [ ]:
gcode = compile_gcode(steps, cfg)
gcode_path = os.path.join(tempfile.gettempdir(), 'demo_output.gcode')
with open(gcode_path, 'w') as f:
    f.write(gcode)
print(f'G-code written to {gcode_path}')
print(f'Lines: {len(gcode.splitlines())}')
print('\nFirst 20 lines:')
print('\n'.join(gcode.splitlines()[:20]))

In [ ]:
stats = dry_run(steps, cfg)
print(f'Path length : {stats.total_path_length_mm:.1f} mm')
print(f'Extrusion   : {stats.total_extrusion_mm:.2f} mm filament')
print(f'B range     : [{stats.b_min_deg:.1f}°, {stats.b_max_deg:.1f}°]')
print(f'Layers      : {stats.layer_count}')
print(f'Moves       : {stats.move_count}')